In [ ]:
````xml
<VSCode.Cell language="markdown">
# Portfolio Analysis
## Stock Market Intelligence Dashboard

**Project**: Stock Market Intelligence Dashboard  
**Purpose**: Analyze portfolio performance, risk, and optimization  
**Date**: March 2026

This notebook analyzes an investment portfolio with the following allocation:
- **Apple (AAPL)**: 40%
- **Microsoft (MSFT)**: 30%
- **Nvidia (NVDA)**: 20%
- **Tesla (TSLA)**: 10%

Initial Investment: $10,000
</VSCode.Cell>

<VSCode.Cell language="code">
# Import required libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import sys
import warnings
warnings.filterwarnings('ignore')

# Add parent directory to path
sys.path.append('../')

# Import custom modules
from utils.helper_functions import load_data, calculate_returns_matrix, calculate_price_matrix
from utils.financial_metrics import (calculate_portfolio_return, 
                                     calculate_portfolio_volatility,
                                     calculate_sharpe_ratio,
                                     calculate_max_drawdown,
                                     calculate_sortino_ratio,
                                     calculate_var, calculate_cvar)

# Display settings
pd.set_option('display.max_columns', None)
%matplotlib inline

print("✓ Libraries imported successfully")
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 1. Load Data

Load the cleaned stock data.
</VSCode.Cell>

<VSCode.Cell language="code">
# Load cleaned data
data = load_data('../data/cleaned/stock_data_cleaned.csv')

if data is not None:
    print(f"✓ Data loaded successfully")
    print(f"Shape: {data.shape}")
    print(f"Symbols: {', '.join(data['symbol'].unique())}")
else:
    print("✗ Failed to load data")
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 2. Define Portfolio

Define portfolio weights and initial investment.
</VSCode.Cell>

<VSCode.Cell language="code">
# Portfolio configuration
INITIAL_INVESTMENT = 10000  # $10,000

portfolio_weights = {
    'AAPL': 0.40,  # 40%
    'MSFT': 0.30,  # 30%
    'NVDA': 0.20,  # 20%
    'TSLA': 0.10   # 10%
}

print("="*80)
print("PORTFOLIO CONFIGURATION")
print("="*80)
print(f"\nInitial Investment: ${INITIAL_INVESTMENT:,.2f}\n")

for symbol, weight in portfolio_weights.items():
    allocation = INITIAL_INVESTMENT * weight
    print(f"{symbol}: {weight*100:.0f}% (${allocation:,.2f})")

print("\n" + "="*80)
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 3. Calculate Initial Shares

Calculate how many shares of each stock to purchase.
</VSCode.Cell>

<VSCode.Cell language="code">
# Get initial prices (first available date for each stock)
initial_shares = {}
initial_prices = {}

print("="*80)
print("INITIAL PURCHASE DETAILS")
print("="*80)

for symbol, weight in portfolio_weights.items():
    symbol_data = data[data['symbol'] == symbol]
    initial_price = symbol_data['close'].iloc[0]
    initial_prices[symbol] = initial_price
    
    allocation = INITIAL_INVESTMENT * weight
    shares = allocation / initial_price
    initial_shares[symbol] = shares
    
    print(f"\n{symbol}:")
    print(f"  Initial Price: ${initial_price:.2f}")
    print(f"  Allocation: ${allocation:,.2f}")
    print(f"  Shares Purchased: {shares:.4f}")

print("\n" + "="*80)
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 4. Calculate Portfolio Value Over Time

Track portfolio value based on stock price changes.
</VSCode.Cell>

<VSCode.Cell language="code">
# Create portfolio value series
price_matrix = calculate_price_matrix(data)

# Calculate individual stock values
portfolio_values = pd.DataFrame(index=price_matrix.index)

for symbol in portfolio_weights.keys():
    if symbol in price_matrix.columns:
        portfolio_values[symbol] = price_matrix[symbol] * initial_shares[symbol]

# Calculate total portfolio value
portfolio_values['total'] = portfolio_values.sum(axis=1)

# Display summary
print("="*80)
print("PORTFOLIO VALUE SUMMARY")
print("="*80)
print(f"\nInitial Value: ${INITIAL_INVESTMENT:,.2f}")
print(f"Current Value: ${portfolio_values['total'].iloc[-1]:,.2f}")
print(f"Total Gain/Loss: ${portfolio_values['total'].iloc[-1] - INITIAL_INVESTMENT:,.2f}")
print(f"Total Return: {((portfolio_values['total'].iloc[-1] / INITIAL_INVESTMENT) - 1) * 100:+.2f}%")
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 5. Portfolio Value Chart

Visualize portfolio growth over time.
</VSCode.Cell>

<VSCode.Cell language="code">
# Plot portfolio value over time
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 10))

# Total portfolio value
ax1.plot(portfolio_values.index, portfolio_values['total'], 
        linewidth=2.5, color='#2E86AB', label='Portfolio Value')
ax1.axhline(y=INITIAL_INVESTMENT, color='red', linestyle='--', 
           linewidth=1.5, label='Initial Investment')
ax1.fill_between(portfolio_values.index, INITIAL_INVESTMENT, 
                 portfolio_values['total'], 
                 where=portfolio_values['total'] >= INITIAL_INVESTMENT,
                 alpha=0.3, color='green', label='Profit')
ax1.fill_between(portfolio_values.index, INITIAL_INVESTMENT, 
                 portfolio_values['total'], 
                 where=portfolio_values['total'] < INITIAL_INVESTMENT,
                 alpha=0.3, color='red', label='Loss')
ax1.set_title('Portfolio Value Over Time', fontsize=16, fontweight='bold')
ax1.set_ylabel('Portfolio Value ($)', fontsize=12)
ax1.legend(loc='best')
ax1.grid(True, alpha=0.3)
ax1.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))

# Individual stock contributions
colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']
for idx, symbol in enumerate(portfolio_weights.keys()):
    if symbol in portfolio_values.columns:
        ax2.plot(portfolio_values.index, portfolio_values[symbol], 
                linewidth=2, color=colors[idx], label=symbol)

ax2.set_title('Individual Stock Values in Portfolio', fontsize=16, fontweight='bold')
ax2.set_xlabel('Date', fontsize=12)
ax2.set_ylabel('Value ($)', fontsize=12)
ax2.legend(loc='best')
ax2.grid(True, alpha=0.3)
ax2.yaxis.set_major_formatter(plt.FuncFormatter(lambda x, p: f'${x:,.0f}'))

plt.tight_layout()
plt.savefig('../data/processed/portfolio_value.png', dpi=300, bbox_inches='tight')
plt.show()
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 6. Portfolio Returns Analysis

Calculate and visualize portfolio returns.
</VSCode.Cell>

<VSCode.Cell language="code">
# Calculate portfolio returns
portfolio_returns = portfolio_values['total'].pct_change().dropna()

# Display statistics
print("="*80)
print("PORTFOLIO RETURN STATISTICS")
print("="*80)
print(f"\nMean Daily Return: {portfolio_returns.mean():.4f}%")
print(f"Median Daily Return: {portfolio_returns.median():.4f}%")
print(f"Std Deviation: {portfolio_returns.std():.4f}%")
print(f"Annualized Return: {portfolio_returns.mean() * 252:.2f}%")
print(f"Annualized Volatility: {portfolio_returns.std() * np.sqrt(252):.2f}%")
print(f"\nBest Day: {portfolio_returns.max():.2f}%")
print(f"Worst Day: {portfolio_returns.min():.2f}%")
print(f"\nPositive Days: {(portfolio_returns > 0).sum()} ({(portfolio_returns > 0).sum() / len(portfolio_returns) * 100:.1f}%)")
print(f"Negative Days: {(portfolio_returns < 0).sum()} ({(portfolio_returns < 0).sum() / len(portfolio_returns) * 100:.1f}%)")
</VSCode.Cell>

<VSCode.Cell language="code">
# Plot return distribution
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histogram
axes[0].hist(portfolio_returns, bins=50, color='#2E86AB', alpha=0.7, edgecolor='black')
axes[0].axvline(portfolio_returns.mean(), color='red', linestyle='--', 
               linewidth=2, label=f'Mean: {portfolio_returns.mean():.4f}%')
axes[0].set_title('Portfolio Daily Returns Distribution', fontweight='bold')
axes[0].set_xlabel('Daily Return (%)')
axes[0].set_ylabel('Frequency')
axes[0].legend()
axes[0].grid(True, alpha=0.3)

# Time series
axes[1].plot(portfolio_returns.index, portfolio_returns, linewidth=1, color='#2E86AB')
axes[1].axhline(y=0, color='black', linestyle='-', linewidth=0.8)
axes[1].set_title('Portfolio Daily Returns Over Time', fontweight='bold')
axes[1].set_xlabel('Date')
axes[1].set_ylabel('Daily Return (%)')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../data/processed/portfolio_returns.png', dpi=300, bbox_inches='tight')
plt.show()
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 7. Risk Metrics

Calculate comprehensive risk metrics for the portfolio.
</VSCode.Cell>

<VSCode.Cell language="code">
# Calculate risk metrics
sharpe_ratio = calculate_sharpe_ratio(portfolio_returns, risk_free_rate=0.02)
sortino_ratio = calculate_sortino_ratio(portfolio_returns, risk_free_rate=0.02)
max_drawdown = calculate_max_drawdown(portfolio_values['total'])
var_95 = calculate_var(portfolio_returns, confidence=0.95)
cvar_95 = calculate_cvar(portfolio_returns, confidence=0.95)

print("="*80)
print("RISK METRICS")
print("="*80)
print(f"\nSharpe Ratio: {sharpe_ratio:.2f}")
print(f"  (Measures risk-adjusted return. Higher is better. > 1 is good, > 2 is very good)")
print(f"\nSortino Ratio: {sortino_ratio:.2f}")
print(f"  (Similar to Sharpe but only considers downside risk)")
print(f"\nMaximum Drawdown: {max_drawdown:.2f}%")
print(f"  (Largest peak-to-trough decline)")
print(f"\nValue at Risk (95% confidence): {var_95:.2f}%")
print(f"  (Expected maximum loss on 95% of days)")
print(f"\nConditional VaR (95% confidence): {cvar_95:.2f}%")
print(f"  (Expected loss on worst 5% of days)")

print("\n" + "="*80)
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 8. Drawdown Analysis

Analyze portfolio drawdowns (declines from peak values).
</VSCode.Cell>

<VSCode.Cell language="code">
# Calculate drawdowns
portfolio_cumulative = portfolio_values['total']
running_max = portfolio_cumulative.expanding().max()
drawdown = (portfolio_cumulative - running_max) / running_max * 100

# Plot drawdowns
plt.figure(figsize=(14, 6))
plt.fill_between(drawdown.index, 0, drawdown, color='red', alpha=0.3)
plt.plot(drawdown.index, drawdown, linewidth=1.5, color='darkred')
plt.axhline(y=0, color='black', linestyle='-', linewidth=0.8)
plt.title('Portfolio Drawdown Over Time', fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Drawdown (%)', fontsize=12)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../data/processed/portfolio_drawdown.png', dpi=300, bbox_inches='tight')
plt.show()

print(f"Maximum Drawdown: {drawdown.min():.2f}%")
print(f"Current Drawdown: {drawdown.iloc[-1]:.2f}%")
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 9. Individual Stock Performance in Portfolio

Analyze how each stock contributed to portfolio performance.
</VSCode.Cell>

<VSCode.Cell language="code">
# Calculate individual stock returns
print("="*80)
print("INDIVIDUAL STOCK PERFORMANCE")
print("="*80)

stock_performance = []

for symbol in portfolio_weights.keys():
    symbol_data = data[data['symbol'] == symbol]
    
    initial_price = symbol_data['close'].iloc[0]
    final_price = symbol_data['close'].iloc[-1]
    
    initial_value = INITIAL_INVESTMENT * portfolio_weights[symbol]
    final_value = portfolio_values[symbol].iloc[-1]
    
    gain_loss = final_value - initial_value
    return_pct = (final_value / initial_value - 1) * 100
    
    contribution = gain_loss / INITIAL_INVESTMENT * 100
    
    stock_performance.append({
        'Symbol': symbol,
        'Weight': f"{portfolio_weights[symbol]*100:.0f}%",
        'Initial Value': f"${initial_value:,.2f}",
        'Final Value': f"${final_value:,.2f}",
        'Gain/Loss': f"${gain_loss:,.2f}",
        'Return': f"{return_pct:+.2f}%",
        'Portfolio Contribution': f"{contribution:+.2f}%"
    })
    
    print(f"\n{symbol}:")
    print(f"  Weight: {portfolio_weights[symbol]*100:.0f}%")
    print(f"  Initial Investment: ${initial_value:,.2f}")
    print(f"  Current Value: ${final_value:,.2f}")
    print(f"  Gain/Loss: ${gain_loss:,.2f}")
    print(f"  Return: {return_pct:+.2f}%")
    print(f"  Contribution to Portfolio: {contribution:+.2f}%")

# Create DataFrame
perf_df = pd.DataFrame(stock_performance)
print("\n" + "="*80)
display(perf_df)
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 10. Portfolio Allocation Pie Chart

Visualize current portfolio allocation by value.
</VSCode.Cell>

<VSCode.Cell language="code">
#Visualize portfolio allocation
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))

# Initial allocation
initial_values = [INITIAL_INVESTMENT * w for w in portfolio_weights.values()]
colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']

ax1.pie(initial_values, labels=portfolio_weights.keys(), autopct='%1.1f%%',
       colors=colors, startangle=90, textprops={'fontsize': 12, 'fontweight': 'bold'})
ax1.set_title('Initial Portfolio Allocation', fontsize=14, fontweight='bold')

# Current allocation
current_values = [portfolio_values[symbol].iloc[-1] for symbol in portfolio_weights.keys()]

ax2.pie(current_values, labels=portfolio_weights.keys(), autopct='%1.1f%%',
       colors=colors, startangle=90, textprops={'fontsize': 12, 'fontweight': 'bold'})
ax2.set_title('Current Portfolio Allocation', fontsize=14, fontweight='bold')

plt.tight_layout()
plt.savefig('../data/processed/portfolio_allocation.png', dpi=300, bbox_inches='tight')
plt.show()
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 11. Correlations and Diversification

Analyze how well the portfolio is diversified.
</VSCode.Cell>

<VSCode.Cell language="code">
# Get returns for each stock
returns_matrix = calculate_returns_matrix(data)
portfolio_stocks = list(portfolio_weights.keys())
portfolio_returns_matrix = returns_matrix[portfolio_stocks]

# Calculate correlation
correlation = portfolio_returns_matrix.corr()

# Visualize correlation
plt.figure(figsize=(8, 6))
sns.heatmap(correlation, annot=True, cmap='coolwarm', center=0, 
           square=True, linewidths=1, cbar_kws={"shrink": 0.8}, 
           fmt='.2f', vmin=-1, vmax=1)
plt.title('Stock Correlation Matrix (Portfolio)', fontsize=16, fontweight='bold')
plt.tight_layout()
plt.savefig('../data/processed/portfolio_correlation.png', dpi=300, bbox_inches='tight')
plt.show()

print("="*80)
print("CORRELATION ANALYSIS")
print("="*80)
print("\nCorrelation Matrix:")
display(correlation)

avg_correlation = correlation.values[np.triu_indices_from(correlation.values, k=1)].mean()
print(f"\nAverage Correlation: {avg_correlation:.2f}")

if avg_correlation < 0.5:
    print("✓ Good diversification (low correlation)")
elif avg_correlation < 0.7:
    print("⚠️ Moderate diversification")
else:
    print("⚠️ Limited diversification (high correlation)")
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 12. Cumulative Return Comparison

Compare portfolio performance against individual stocks.
</VSCode.Cell>

<VSCode.Cell language="code">
# Calculate cumulative returns
portfolio_cumulative_return = (portfolio_values['total'] / INITIAL_INVESTMENT - 1) * 100

plt.figure(figsize=(14, 8))

# Plot portfolio
plt.plot(portfolio_values.index, portfolio_cumulative_return, 
        linewidth=3, color='black', label='Portfolio', linestyle='-')

# Plot individual stocks
colors = ['#2E86AB', '#A23B72', '#F18F01', '#C73E1D']
for idx, symbol in enumerate(portfolio_weights.keys()):
    symbol_data = data[data['symbol'] == symbol]
    cumulative = symbol_data['cumulative_return'] * 100
    plt.plot(symbol_data['date'], cumulative, 
            linewidth=2, color=colors[idx], label=symbol, alpha=0.7)

plt.axhline(y=0, color='gray', linestyle='--', linewidth=1)
plt.title('Cumulative Returns: Portfolio vs Individual Stocks', fontsize=16, fontweight='bold')
plt.xlabel('Date', fontsize=12)
plt.ylabel('Cumulative Return (%)', fontsize=12)
plt.legend(loc='best', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../data/processed/portfolio_vs_stocks.png', dpi=300, bbox_inches='tight')
plt.show()
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 13. Portfolio Performance Summary

Generate a comprehensive summary report.
</VSCode.Cell>

<VSCode.Cell language="code">
# Create summary report
summary = {
    'Metric': [
        'Initial Investment',
        'Current Value',
        'Total Gain/Loss',
        'Total Return',
        'Annualized Return',
        'Annualized Volatility',
        'Sharpe Ratio',
        'Sortino Ratio',
        'Maximum Drawdown',
        'Best Day',
        'Worst Day',
        'VaR (95%)',
        'CVaR (95%)',
        'Positive Days',
        'Average Correlation'
    ],
    'Value': [
        f"${INITIAL_INVESTMENT:,.2f}",
        f"${portfolio_values['total'].iloc[-1]:,.2f}",
        f"${portfolio_values['total'].iloc[-1] - INITIAL_INVESTMENT:+,.2f}",
        f"{((portfolio_values['total'].iloc[-1] / INITIAL_INVESTMENT) - 1) * 100:+.2f}%",
        f"{portfolio_returns.mean() * 252:.2f}%",
        f"{portfolio_returns.std() * np.sqrt(252):.2f}%",
        f"{sharpe_ratio:.2f}",
        f"{sortino_ratio:.2f}",
        f"{max_drawdown:.2f}%",
        f"{portfolio_returns.max():+.2f}%",
        f"{portfolio_returns.min():+.2f}%",
        f"{var_95:.2f}%",
        f"{cvar_95:.2f}%",
        f"{(portfolio_returns > 0).sum() / len(portfolio_returns) * 100:.1f}%",
        f"{avg_correlation:.2f}"
    ]
}

summary_df = pd.DataFrame(summary)

print("="*80)
print("PORTFOLIO PERFORMANCE SUMMARY")
print("="*80)
display(summary_df)

# Export summary
summary_df.to_csv('../data/processed/portfolio_summary.csv', index=False)
print("\n✓ Portfolio summary exported to: ../data/processed/portfolio_summary.csv")
</VSCode.Cell>

<VSCode.Cell language="markdown">
## 14. Export Portfolio Data

Export portfolio data for Tableau dashboard.
</VSCode.Cell>

<VSCode.Cell language="code">
# Prepare portfolio data for export
portfolio_export = portfolio_values.reset_index()
portfolio_export.columns = ['date'] + list(portfolio_export.columns[1:])

# Add returns
portfolio_export['daily_return'] = portfolio_returns.values
portfolio_export = portfolio_export.fillna(0)

# Export
portfolio_export.to_csv('../data/processed/portfolio_timeseries.csv', index=False)
print("✓ Portfolio timeseries exported to: ../data/processed/portfolio_timeseries.csv")

# Export stock performance summary
perf_df.to_csv('../data/processed/stock_performance_summary.csv', index=False)
print("✓ Stock performance summary exported to: ../data/processed/stock_performance_summary.csv")
</VSCode.Cell>

<VSCode.Cell language="markdown">
## Conclusion

### Portfolio Analysis Summary:

**Investment Performance**:
- The portfolio allocation strategy balanced growth and stability
- Diversification across tech sectors provided risk mitigation
- Returns were influenced by individual stock performance weighted by allocation

**Risk Assessment**:
- Sharpe Ratio indicates risk-adjusted performance quality
- Maximum Drawdown shows potential downside risk
- Value at Risk provides probabilistic loss estimates

**Key Insights**:
1. **40% AAPL allocation** provided stable returns as the largest position
2. **Stock correlations** show how movements are interconnected
3. **Volatility patterns** indicate periods of higher uncertainty
4. **Cumulative returns** demonstrate the power of long-term investing

**Recommendations**:
- Monitor correlation levels for diversification effectiveness
- Rebalance periodically to maintain target weights
- Consider adding assets with lower correlations for better diversification
- Set stop-loss levels based on maximum drawdown tolerance

---
**End of Portfolio Analysis**
</VSCode.Cell>
````